In [ ]:
# from gliner import GLiNER
# from transformers import AutoTokenizer

# # Load a GLiNER model
# model = GLiNER.from_pretrained("/kaggle/working/GLiNER/models/checkpoint-15950")
# tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/GLiNER/models/checkpoint-15950")

# from huggingface_hub import login

# login("hf_")
# model.push_to_hub("HinoEiji/GLiNER-cafebert")
# tokenizer.push_to_hub("HinoEiji/GLiNER-cafebert")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/HinoEiji/GLiNER-cafebert/commit/9bce4ffce754f6290ac193ce19a0f2d53fbfe0a2', commit_message='Upload tokenizer', commit_description='', oid='9bce4ffce754f6290ac193ce19a0f2d53fbfe0a2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/HinoEiji/GLiNER-cafebert', endpoint='https://huggingface.co', repo_type='model', repo_id='HinoEiji/GLiNER-cafebert'), pr_revision=None, pr_num=None)

In [25]:
from seqeval.metrics.sequence_labeling import get_entities
ent_map = {
    "PRODUCT" : "sản phẩm",
    "PRODUCT FEATURE" : "đặc trưng sản phẩm",
    "PRODUCT USAGE" : "công dụng sản phẩm",
    "PRODUCT QUALITY" : "chất lượng sản phẩm",
    "PRODUCT DESIGN" : "thiết kế sản phẩm",
    "PRICE" : "giá cả",
    "SERVICE" : "dịch vụ",
    "BRANDING" : "thương hiệu",
    "GENERAL" : "chung",
    "DELIVERY" : "giao hàng"
}
def read_conll_file(file_path, ent_path = None):
    data = []
    
    tokens = []
    tags = []

    if ent_path:
        try:
            with open(ent_path, "r", encoding="utf-8") as f:
                list_ent = [line.strip() for line in f if line.strip()]
                list_ent = [ent_map[ent] for ent in list_ent]
        except:
            list_ent = None
    else:
        list_ent = None
    
    if "train" not in file_path:
        list_ent = None
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            # gặp dòng trống => kết thúc 1 sample
            if not line:
                if tokens:
                    data.append(convert_to_span(tokens, tags, list_ent))
                    tokens, tags = [], []
                continue
            
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            
            token, tag = parts
            tokens.append(token)
            tags.append(tag)
    
    # case file không kết thúc bằng dòng trống
    if tokens:
        data.append(convert_to_span(tokens, tags, list_ent))
    
    return data

def convert_to_span(tokens, tags, list_ent = None):
    entities = get_entities(tags)
    new_tags = []
    for tag in tags:
        if tag == "O":
            new_tags.append(tag)
            continue
        splits = tag.split("-")
        new_tags.append(splits[0]+"-"+ent_map[splits[1]])
    
    ner = []
    ner_labels = []

    for ent_type, start, end in entities:
        ent_type = ent_map[ent_type]
        ner.append([start,end, ent_type])
        ner_labels.append(ent_type)
    if list_ent:
        ner_labels = set(ner_labels)
        ner_negatives = set(list_ent) - ner_labels

        return{
            "tokenized_text": tokens,
            "ner": ner,
            "ner_labels": list(ner_labels),
            "ner_negatives": list(ner_negatives),
            "tags" : new_tags
        }
        

    return {
        "tokenized_text": tokens,
        "ner": ner,
        "tags" : new_tags
    }

In [26]:
gt=read_conll_file("/kaggle/working/GLiNER/custom_train_data/v3.3.1/test.txt")

In [27]:
# y_true = []
# y_pred = []
# for i in range(len(gt)):
#     text = " ".join(gt[i]['tokenized_text'])
#     entities = model.predict_entities(text, labels, threshold=0.9)
#     _, pred = char_to_bio(text, entities)
#     gt[i]['pred'] = pred
#     y_true.append(gt[i]['tags'])
#     y_pred.append(pred)

In [28]:
import json
with open("/kaggle/working/GLiNER/cafebert-results-checkpoint-15950_threshold_0.9.json", "r") as f:
    prediction = json.load(f)
    preds = prediction['my_data']['predictions']

In [29]:
gt[0]['tokenized_text'][0:6+1]

['đã', 'dùng', 'và', 'thấy', 'rất', 'hài', 'lòng,']

In [30]:
y_true = []
y_pred = []
for i in range(len(gt)):
    tokens = gt[i]["tokenized_text"]
    pred_tag = ["O"]*len(tokens)
    for pred in preds[i]:
        pred_tag[pred["start"]]="B-"+pred['label']
        I = ["I-"+pred['label']]*(pred["end"]-pred["start"])
        pred_tag[pred["start"]+1 : pred["end"]+1] = I
    y_true.append(gt[i]['tags'])
    y_pred.append(pred_tag)
        

In [31]:
from seqeval.metrics import classification_report

In [32]:
print(classification_report(y_true, y_pred, zero_division=0, digits =4))

                     precision    recall  f1-score   support

              chung     0.4973    0.5962    0.5423       312
chất lượng sản phẩm     0.6416    0.4531    0.5311       320
 công dụng sản phẩm     0.5697    0.4388    0.4957       531
            dịch vụ     0.5269    0.2722    0.3590       180
          giao hàng     0.7172    0.6311    0.6714       450
             giá cả     0.6541    0.6000    0.6259       145
           sản phẩm     0.4924    0.4121    0.4487       313
  thiết kế sản phẩm     0.5968    0.4548    0.5162       332
        thương hiệu     0.1005    0.4889    0.1667        45
 đặc trưng sản phẩm     0.3323    0.3675    0.3490       283

          micro avg     0.5190    0.4775    0.4974      2911
          macro avg     0.5129    0.4715    0.4706      2911
       weighted avg     0.5586    0.4775    0.5077      2911



In [33]:
y_true_flat = [tag for sent in y_true for tag in sent]
y_pred_flat = [tag for sent in y_pred for tag in sent]

In [34]:
from sklearn.metrics import classification_report

print(classification_report(y_true_flat, y_pred_flat, digits = 4))

                       precision    recall  f1-score   support

              B-chung     0.5187    0.6218    0.5656       312
B-chất lượng sản phẩm     0.7168    0.5062    0.5934       320
 B-công dụng sản phẩm     0.6406    0.4934    0.5574       531
            B-dịch vụ     0.5699    0.2944    0.3883       180
          B-giao hàng     0.7727    0.6800    0.7234       450
             B-giá cả     0.6917    0.6345    0.6619       145
           B-sản phẩm     0.5573    0.4665    0.5078       313
  B-thiết kế sản phẩm     0.6957    0.5301    0.6017       332
        B-thương hiệu     0.1142    0.5556    0.1894        45
 B-đặc trưng sản phẩm     0.3802    0.4205    0.3993       283
              I-chung     0.4471    0.3799    0.4107      1390
I-chất lượng sản phẩm     0.7510    0.4318    0.5483      1334
 I-công dụng sản phẩm     0.6588    0.4450    0.5312      2634
            I-dịch vụ     0.5626    0.2485    0.3448      1030
          I-giao hàng     0.8129    0.5983    0.6893  

In [35]:
y_t = [tag.split("-")[0] for tag in y_true_flat]
y_p = [tag.split("-")[0] for tag in y_pred_flat]
print(classification_report(y_t, y_p, digits = 4))

              precision    recall  f1-score   support

           B     0.7827    0.7200    0.7500      2911
           I     0.8332    0.6331    0.7195     12805
           O     0.4684    0.7086    0.5640      6454

    accuracy                         0.6665     22170
   macro avg     0.6948    0.6872    0.6779     22170
weighted avg     0.7204    0.6665    0.6782     22170



In [36]:
y_span_t = []
y_span_p = []
for i in range(len(y_true)):
    tags = []
    for tag in y_true[i]:
        if tag !="O":
            tag = tag[:2] + "SPAN"
        tags.append(tag)
    y_span_t.append(tags)

for i in range(len(y_pred)):
    tags = []
    for tag in y_pred[i]:
        if tag !="O":
            tag = tag[:2] + "SPAN"
        tags.append(tag)
    y_span_p.append(tags)      
        

In [37]:
from seqeval.metrics import classification_report
print(classification_report(y_span_t, y_span_p, zero_division=0, digits =4))

              precision    recall  f1-score   support

        SPAN     0.6886    0.6335    0.6599      2911

   micro avg     0.6886    0.6335    0.6599      2911
   macro avg     0.6886    0.6335    0.6599      2911
weighted avg     0.6886    0.6335    0.6599      2911

